# Lesson 1 exercise — Tokens in, tokens out

**Goal:** load a small chat model, see *exactly* what it reads, and watch it fail at the JSON task you'll fine-tune it for later.

**How to use this notebook**
1. In Colab: *Runtime → Change runtime type → T4 GPU* (optional — this model is tiny and also runs on CPU).
2. Run cells top to bottom. Wherever you see `...` after a `# TODO`, replace it with your code.
3. Run the **check** cell under each TODO. ✅ means you've got it; ❌ gives you a hint.
4. Try each TODO yourself before using a hint. The effort is what makes it stick.

Stuck? Look up the error in the course's troubleshooting sheet (`reference/troubleshooting.html`). Still stuck? Ask on the Hugging Face Forums with your code and the full traceback.

## 0 · Setup
Installs the library versions this course uses (about a minute), and defines `device` and the `check` helper.

In [ ]:
%pip install -q -U "transformers>=5.19,<6" "trl>=1.15,<2" "peft>=0.21,<1" "datasets>=5,<6" "accelerate>=1.15,<2"
import torch, transformers
device = "cuda" if torch.cuda.is_available() else "cpu"
print("transformers", transformers.__version__, "| torch", torch.__version__, "| device:", device)

def check(passed, ok_msg, hint):
    print(("✅ " + ok_msg) if passed else ("❌ " + hint))

## 1 · Load the tokenizer and the model — a matched pair
Same idea as `keras.applications.ResNet50(weights="imagenet")` plus its `preprocess_input`: the tokenizer and the model come from the **same checkpoint id**, and they only work together.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "HuggingFaceTB/SmolLM2-135M-Instruct"

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float32).to(device)  # full precision: tiny model, runs anywhere

print("Auto picked:", type(tok).__name__, "+", type(model).__name__)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters, dtype {model.dtype}, on {device}")

## 2 · The chat template: what the model *actually* reads
A chat model never sees your list of messages. It sees **one string** with special tokens marking who is speaking. The tokenizer carries a template that builds that string.

**TODO 1:** Render `messages` into that string. You want **text, not token IDs**, and the template should end by opening the assistant's turn so the model knows it's its turn to speak.

In [ ]:
messages = [
    {"role": "user", "content": "Extract the person's name and city as JSON: 'Hi, I'm Sara from Cairo.'"},
]

# TODO 1
text = ...

print(text)

In [ ]:
# check 1
check(isinstance(text, str) and text.rstrip().endswith("assistant"),
      "Rendered as text, ending with the assistant turn.",
      "Use tok.apply_chat_template(...). Look at its `tokenize` and `add_generation_prompt` arguments.")

**Look closely at the output above.**
- Which special tokens mark the start and end of each turn?
- There's a **system message you didn't write**. Where did it come from?

Keep both in mind. When you fine-tune, your training data must go through *this same template*.

## 3 · From text to tensors
**TODO 2:** Make the same call, but this time **tokenize** and return **PyTorch tensors** as a dict (you want both `input_ids` and `attention_mask`).

In [ ]:
# TODO 2
inputs = ...

inputs = inputs.to(device)          # PyTorch: inputs must live on the same device as the model
print(list(inputs.keys()))
print("input_ids shape:", tuple(inputs["input_ids"].shape))   # (batch, sequence_length)

In [ ]:
# check 2
check(hasattr(inputs, "keys") and "input_ids" in inputs and "attention_mask" in inputs and inputs["input_ids"].ndim == 2,
      f"Got a batch of 1 sequence with {inputs['input_ids'].shape[1]} tokens." if hasattr(inputs, "keys") and "input_ids" in inputs else "",
      "Same call as TODO 1, but drop tokenize=False and ask for return_tensors='pt' and return_dict=True.")

Peek at the tokens. `Ġ` marks a leading space and `Ċ` is a newline. This is how byte-level BPE writes them.

In [ ]:
tokens = tok.convert_ids_to_tokens(inputs["input_ids"][0])
print(len(tokens), "tokens:")
print(tokens)

## 4 · One forward pass → logits
`model(**inputs)` is a single forward pass, like calling a Keras model on a batch.
`torch.no_grad()` tells PyTorch not to record operations for backprop. You don't need gradients here, so it saves memory.

The output `logits` has shape **(batch, sequence, vocab)**. At *every* position, the model scores every token in its vocabulary as the possible next token.

**TODO 3:** Get the id of the single most likely **next** token after the prompt (greedy decoding).
*Hint:* which position in the sequence predicts what comes *after* the prompt? Then take the argmax over the vocabulary.

In [ ]:
with torch.no_grad():
    out = model(**inputs)
logits = out.logits
print("logits shape:", tuple(logits.shape))

# TODO 3
next_id = ...

print("next token:", repr(tok.decode([int(next_id)])))

In [ ]:
# check 3 — generate() with 1 new token and no sampling does the same thing
with torch.no_grad():
    expected = model.generate(**inputs, max_new_tokens=1, do_sample=False)[0, -1].item()
check(int(next_id) == expected,
      "Matches generate()'s first token. generate() is just this step on a loop.",
      "Take the LAST position of sequence 0, then argmax over the vocab dimension.")

## 5 · Generate a full reply
`generate()` runs that step over and over: predict a token, append it, predict again. It stops at the end-of-turn token or at `max_new_tokens`.
It returns **the prompt plus the new tokens**.

**TODO 4:** Decode **only the newly generated tokens** into a string, without special tokens.

In [ ]:
with torch.no_grad():
    out_ids = model.generate(**inputs, max_new_tokens=80, do_sample=False)

prompt_len = inputs["input_ids"].shape[1]
print("prompt tokens:", prompt_len, "| total tokens:", out_ids.shape[1])

# TODO 4
reply = ...

print(reply)

In [ ]:
# check 4
check(isinstance(reply, str) and len(reply.strip()) > 0
      and "You are a helpful AI assistant" not in reply and "<|im_" not in reply,
      "Clean reply: no prompt echoed, no special tokens.",
      "Slice out_ids[0] from prompt_len onwards, then tok.decode(..., skip_special_tokens=True).")

## 6 · Did it do the job?
Look at `reply`. Did you get valid JSON like `{"name": "Sara", "city": "Cairo"}`? Run this to find out:

In [ ]:
import json
try:
    print("Valid JSON ✅", json.loads(reply))
except json.JSONDecodeError:
    print("Not valid JSON ❌ — this is the gap your capstone fine-tune will close.")

## Stretch (optional, ~5 min)
1. Add a system message: `{"role": "system", "content": "You reply with JSON only."}` at the start of `messages`, then rerun sections 2–6. Does prompting alone fix it?
2. Set `do_sample=True, temperature=0.7` in `generate()` and run it 5 times. How often do you get valid JSON?

Write down what you saw. You'll compare it against your fine-tuned model in the capstone.